In [ ]:
from loguru import logger
import functools
import time

def log_execution(func):
    """记录函数执行的装饰器"""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger.debug(f"开始执行: {func.__name__}")
        start = time.time()

        try:
            result = func(*args, **kwargs)
            duration = (time.time() - start) * 1000
            logger.success(f"执行完成: {func.__name__} | 耗时: {duration:.2f}ms")
            return result
        except Exception as e:
            duration = (time.time() - start) * 1000
            logger.error(f"执行失败: {func.__name__} | 耗时: {duration:.2f}ms | 错误: {e}")
            raise

    return wrapper

# 使用装饰器
@log_execution
def node_a(state, runtime):
    return {"node_a_output": "node_a is running"}

In [ ]:
# config/logging.py
from loguru import logger
import sys
import os
from pathlib import Path

def setup_logging(env: str = "development"):
    """设置日志配置"""

    # 创建日志目录
    log_dir = Path("logs")
    log_dir.mkdir(exist_ok=True)

    # 移除默认 handler
    logger.remove()

    # 根据环境设置日志级别
    level = "DEBUG" if env == "development" else "INFO"

    # 控制台输出 - 彩色美化
    logger.add(
        sys.stdout,
        format=(
            "<green>{time:YYYY-MM-DD HH:mm:ss}</green> | "
            "<level>{level: <8}</level> | "
            "<cyan>{name}</cyan>:<cyan>{function}</cyan>:<cyan>{line}</cyan> | "
            "<level>{message}</level>"
        ),
        colorize=True,
        level=level,
        backtrace=True,
        diagnose=env == "development"
    )

    # 文件输出 - JSON 结构化
    logger.add(
        log_dir / "app_{time:YYYY-MM-DD}.json.log",
        format="{message}",
        serialize=True,
        level="INFO",
        rotation="100 MB",
        retention="30 days",
        compression="gz"
    )

    # 文件输出 - 详细文本
    logger.add(
        log_dir / "app_{time:YYYY-MM-DD}.log",
        format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {name}:{function}:{line} | {message}",
        level=level,
        rotation="100 MB",
        retention="30 days",
        compression="zip"
    )

    # 错误日志单独存储
    logger.add(
        log_dir / "errors_{time:YYYY-MM-DD}.log",
        format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {name}:{function}:{line} | {message}",
        level="ERROR",
        rotation="50 MB",
        retention="90 days",
        backtrace=True,
        diagnose=True
    )

    # 添加默认的上下文信息
    logger.configure(
        extra={
            "app": "langgraph",
            "environment": env,
            "hostname": os.uname().nodename if hasattr(os, 'uname') else "unknown"
        }
    )

    logger.info(f"日志系统初始化完成 | 环境: {env} | 日志级别: {level}")
    return logger

# 在应用入口调用
logger = setup_logging(env="development")

In [ ]:
from loguru import logger
import sys
from datetime import datetime

logger.remove()

# 自定义格式函数
def custom_format(record):
    # 根据日志级别改变颜色
    colors = {
        "DEBUG": "<cyan>",
        "INFO": "<green>",
        "SUCCESS": "<bright-green>",
        "WARNING": "<yellow>",
        "ERROR": "<red>",
        "CRITICAL": "<red><bright>"
    }

    level_color = colors.get(record["level"].name, "")
    reset = "</>" if level_color else ""

    return (
        f"{level_color}[{record['time']:HH:mm:ss.SSS}]</> | "
        f"{level_color}{record['level'].name: <8}</> | "
        f"{record['name']}.{record['function']}:{record['line']} | "
        f"{record['message']}\n"
    )

logger.add(sys.stdout, format=custom_format, colorize=True, level="DEBUG")

# 添加额外的信息到日志上下文
ctx = logger.bind(request_id="12345", user_id="user_001")
ctx.info("处理请求")